# S5 Causal Masked Reconstruction Colab

This notebook trains a causal `S5` encoder with masked signal reconstruction on Utah-array cache shards stored in Google Drive.

Default design:

- reconstruct normalized raw patch values in `TX` space
- causal `S5` backbone with session-keyed token read-in / read-out boundaries
- patch-level masking by default with `patch_size=4`, `patch_stride=2`
- contiguous masked spans with a fixed overall mask ratio
- same held-out `Brain2Text25` frozen phoneme probe workflow used in the other `s5` notebooks


In [ ]:
# Mount Drive and resolve cache / output roots.

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")
cache_candidates = [
    DRIVE_ROOT / "utah_ssl" / "data" / "cache_v1",
]

CACHE_ROOT = next((p for p in cache_candidates if p.exists()), cache_candidates[0])
OUTPUT_ROOT = DRIVE_ROOT / "utah_ssl" / "outputs" / "ssl_experiments" / "masked_reconstruction"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("DRIVE_ROOT :", DRIVE_ROOT)
print("CACHE_ROOT :", CACHE_ROOT, "| exists:", CACHE_ROOT.exists())
print("OUTPUT_ROOT:", OUTPUT_ROOT, "| exists:", OUTPUT_ROOT.exists())

if CACHE_ROOT.exists():
    datasets = sorted(p.name for p in CACHE_ROOT.iterdir() if p.is_dir())
    print("datasets:", datasets)
else:
    print("cache candidates checked:")
    for path in cache_candidates:
        print(" -", path)


In [ ]:
# Clone the public repo and import the reusable masked SSL helpers.

import os
import subprocess
import sys

REPO_URL = "https://github.com/ethan-read/utah-ssl.git"
REPO_DIR = Path("/content/utah-ssl")
EXPERIMENTS_DIR = REPO_DIR / "analysis" / "active" / "ssl_experiments"
MASKED_SSL_DIR = EXPERIMENTS_DIR / "masked_ssl"
SSL_DIR = REPO_DIR / "analysis" / "active" / "transfer_benchmark" / "ssl_autoresearch"

os.chdir("/content")

if REPO_DIR.exists():
    print("Using existing repo:", REPO_DIR)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)

for candidate in (REPO_DIR, EXPERIMENTS_DIR, SSL_DIR):
    candidate_str = str(candidate)
    if candidate_str not in sys.path:
        sys.path.insert(0, candidate_str)

os.environ["SSL_AUTORESEARCH_CACHE_ROOT"] = str(CACHE_ROOT)
os.environ["SSL_AUTORESEARCH_OUTPUT_ROOT"] = str(OUTPUT_ROOT)

if not MASKED_SSL_DIR.exists():
    raise FileNotFoundError(
        "The cloned repo does not contain analysis/active/ssl_experiments/masked_ssl. "
        "Make sure REPO_DIR points at a checkout that includes the masked reconstruction package."
    )

from masked_ssl import (
    CacheAccessConfig,
    DownstreamProbeConfig,
    SSLTrainingConfig,
    build_random_init_probe_state,
    build_segment_sampler,
    list_ssl_checkpoints,
    load_precomputed_session_feature_stats_into_cache_context,
    plot_ssl_training_history,
    prepare_cache_context,
    recover_downstream_probe_state,
    recover_ssl_run_state_from_checkpoint,
    resolve_ssl_checkpoint_path,
    run_checkpoint_probe_suite,
    run_downstream_probe,
    run_probe_head_sweep,
    resume_ssl_training,
    run_ssl_training,
    display_probe_summaries,
    display_ssl_reconstruction_report,
    SWEEP_VITAL_COLUMNS,
    run_sigma_mask_probe_sweep,
)
from masked_ssl.probe import build_downstream_probe_problem

print("cwd:", Path.cwd())
print("repo dir exists:", REPO_DIR.exists(), REPO_DIR)
print("experiments dir exists:", EXPERIMENTS_DIR.exists(), EXPERIMENTS_DIR)
print("masked_ssl dir exists:", MASKED_SSL_DIR.exists(), MASKED_SSL_DIR)
print("ssl dir exists:", SSL_DIR.exists(), SSL_DIR)
print("SSL_AUTORESEARCH_CACHE_ROOT:", os.environ["SSL_AUTORESEARCH_CACHE_ROOT"])
print("SSL_AUTORESEARCH_OUTPUT_ROOT:", os.environ["SSL_AUTORESEARCH_OUTPUT_ROOT"])


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

Path("/content/drive/MyDrive/utah_ssl/scripts").mkdir(parents=True, exist_ok=True)


In [ ]:
%%writefile /content/drive/MyDrive/utah_ssl/scripts/build_smoothed_cache.py
print("hello from build_smoothed_cache")


In [ ]:
!ls -l /content/drive/MyDrive/utah_ssl/scripts/build_smoothed_cache.py
!python /content/drive/MyDrive/utah_ssl/scripts/build_smoothed_cache.py


In [ ]:
%%writefile /content/drive/MyDrive/utah_ssl/scripts/build_smoothed_cache.py
"""Build a sibling pre-smoothed cache root from a canonical Utah SSL cache."""
from __future__ import annotations
import argparse, json, shutil
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Sequence
import numpy as np
import torch
import torch.nn.functional as F

def _gaussian_kernel_1d(sigma: float, *, device: torch.device, dtype: torch.dtype, radius: int | None = None) -> torch.Tensor:
    sigma = float(sigma)
    if sigma <= 0.0:
        return torch.ones((1,), device=device, dtype=dtype)
    effective_radius = int(radius) if radius is not None else max(1, int(np.ceil(4.0 * sigma)))
    positions = torch.arange(-effective_radius, effective_radius + 1, device=device, dtype=dtype)
    kernel = torch.exp(-0.5 * (positions / sigma).pow(2))
    return kernel / kernel.sum().clamp_min(1e-8)

def _apply_gaussian_smoothing(x_seq: torch.Tensor, feature_mask: torch.Tensor, *, sigma_bins: float) -> torch.Tensor:
    sigma = float(sigma_bins)
    if sigma <= 0.0 or x_seq.shape[0] <= 1:
        return x_seq
    present_idx = torch.nonzero(feature_mask.bool(), as_tuple=False).squeeze(1)
    if present_idx.numel() == 0:
        return x_seq
    max_reflect_radius = int(x_seq.shape[0] - 1)
    if max_reflect_radius <= 0:
        return x_seq
    kernel_radius = min(max(1, int(np.ceil(4.0 * sigma))), max_reflect_radius)
    kernel = _gaussian_kernel_1d(sigma, device=x_seq.device, dtype=x_seq.dtype, radius=kernel_radius)
    kernel = kernel / kernel.sum().clamp_min(1e-8)
    selected = x_seq[:, present_idx].transpose(0, 1).unsqueeze(0)
    padded = F.pad(selected, (kernel_radius, kernel_radius), mode="reflect")
    weight = kernel.view(1, 1, -1).expand(selected.shape[1], 1, -1)
    smoothed = F.conv1d(padded, weight, groups=selected.shape[1]).squeeze(0).transpose(0, 1)
    out = x_seq.clone()
    out[:, present_idx] = smoothed
    return out

def _timestamp_utc() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def _iter_dataset_names(src_root: Path, requested: Sequence[str] | None) -> list[str]:
    if requested:
        for d in requested:
            if not (src_root / d).is_dir():
                raise FileNotFoundError(f"Dataset not found: {src_root / d}")
        return [str(d) for d in requested]
    return sorted(p.name for p in src_root.iterdir() if p.is_dir())

def _iter_shard_dirs(dataset_root: Path) -> list[Path]:
    shard_parent = dataset_root / "shards"
    if shard_parent.is_dir():
        return sorted(p for p in shard_parent.iterdir() if p.is_dir())
    return sorted(p for p in dataset_root.iterdir() if p.is_dir() and p.name != "shards")

def smooth_feature_array(array: np.ndarray, *, sigma_bins: float) -> np.ndarray:
    if array.ndim != 2:
        raise ValueError(f"Expected rank-2 feature array, got {array.shape}")
    if array.size == 0 or float(sigma_bins) <= 0.0:
        return np.array(array, copy=True)
    tensor = torch.from_numpy(np.asarray(array, dtype=np.float32))
    feature_mask = torch.ones(int(tensor.shape[1]), dtype=torch.float32)
    smoothed = _apply_gaussian_smoothing(tensor, feature_mask, sigma_bins=float(sigma_bins))
    return smoothed.cpu().numpy().astype(array.dtype, copy=False)

def build_smoothed_cache(*, src_root: str | Path, dst_root: str | Path, sigma_bins: float, datasets: Sequence[str] | None = None, overwrite: bool = False, dry_run: bool = False) -> dict[str, Any]:
    src_root = Path(src_root)
    dst_root = Path(dst_root)
    sigma_bins = float(sigma_bins)
    if sigma_bins <= 0.0:
        raise ValueError("sigma_bins must be positive")
    if not src_root.is_dir():
        raise FileNotFoundError(f"Missing source cache root: {src_root}")
    if src_root.resolve() == dst_root.resolve():
        raise ValueError("dst_root must differ from src_root")

    dataset_names = _iter_dataset_names(src_root, datasets)
    if dst_root.exists() and not overwrite and not dry_run:
        raise FileExistsError(f"Destination exists: {dst_root}")
    if dst_root.exists() and overwrite and not dry_run:
        shutil.rmtree(dst_root)
    if not dry_run:
        dst_root.mkdir(parents=True, exist_ok=True)

    summary = {"src_root": str(src_root), "dst_root": str(dst_root), "sigma_bins": sigma_bins, "datasets": {}, "dry_run": bool(dry_run)}
    for dataset in dataset_names:
        src_ds = src_root / dataset
        dst_ds = dst_root / dataset
        shard_dirs = _iter_shard_dirs(src_ds)
        manifest = src_ds / "manifest.jsonl"
        metadata = src_ds / "metadata.json"
        if not manifest.exists() or not metadata.exists():
            raise FileNotFoundError(f"Missing manifest/metadata for {dataset}")

        if not dry_run:
            dst_ds.mkdir(parents=True, exist_ok=True)
            shutil.copy2(manifest, dst_ds / "manifest.jsonl")

        meta = json.loads(metadata.read_text())
        meta["smoothing_provenance"] = {
            "source_cache_root": str(src_root.resolve()),
            "sigma_bins": sigma_bins,
            "implementation": "local_colab_build_smoothed_cache.py",
            "created_utc": _timestamp_utc(),
        }
        if not dry_run:
            (dst_ds / "metadata.json").write_text(json.dumps(meta, indent=2))

        files_copied = 0
        feature_files_smoothed = 0
        for src_shard in shard_dirs:
            rel = src_shard.relative_to(src_ds)
            dst_shard = dst_ds / rel
            if not dry_run:
                dst_shard.mkdir(parents=True, exist_ok=True)
            for src_file in sorted(src_shard.iterdir()):
                if not src_file.is_file():
                    continue
                dst_file = dst_shard / src_file.name
                if src_file.name in {"tx.npy", "sbp.npy"}:
                    feature_files_smoothed += 1
                    if not dry_run:
                        arr = np.load(src_file)
                        np.save(dst_file, smooth_feature_array(arr, sigma_bins=sigma_bins))
                else:
                    files_copied += 1
                    if not dry_run:
                        shutil.copy2(src_file, dst_file)

        summary["datasets"][dataset] = {
            "manifest_copied": True,
            "metadata_copied": True,
            "shard_count": len(shard_dirs),
            "files_copied": files_copied,
            "feature_files_smoothed": feature_files_smoothed,
        }
    return summary

if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--src", required=True)
    p.add_argument("--dst", required=True)
    p.add_argument("--sigma-bins", required=True, type=float)
    p.add_argument("--datasets", nargs="+", default=None)
    p.add_argument("--overwrite", action="store_true")
    p.add_argument("--dry-run", action="store_true")
    args = p.parse_args()
    out = build_smoothed_cache(
        src_root=args.src,
        dst_root=args.dst,
        sigma_bins=args.sigma_bins,
        datasets=args.datasets,
        overwrite=args.overwrite,
        dry_run=args.dry_run,
    )
    print(json.dumps(out, indent=2))


In [ ]:
!python /content/drive/MyDrive/utah_ssl/scripts/build_smoothed_cache.py \
  --src /content/drive/MyDrive/utah_ssl/data/cache_v1 \
  --dst /content/drive/MyDrive/utah_ssl/data/cache_v1_smoothed_sigma2p0 \
  --sigma-bins 2.0 \
  --datasets brain2text25 \
  --overwrite


In [ ]:
import json
from pathlib import Path

meta = Path("/content/drive/MyDrive/utah_ssl/data/cache_v1_smoothed_sigma2p0/brain2text24/metadata.json")
m = json.loads(meta.read_text())
print(m.get("smoothing_provenance", {}))


In [ ]:
# Experiment config.

SEED = 7

SSL_STATE_MODE = "recover"  # Set to "recover" to load a previous checkpoint.
GAUSSIAN_SMOOTHING_SIGMA_BINS = 2.0
EXPECTED_SESSION_STATS_BIN_STRIDE = 2  # frozen for cached session-stats provenance
SESSION_STATS_VARIANT = "smoothed"  # one of {"smoothed", "unsmoothed", "none"}
STABLE_SSL_SESSION_STATS_PATH_UNSMOOTHED = Path("/content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/contrastive/precomputed_ssl_session_stats/session_feature_stats_session_featurewise_v1_refds000950_cap126682_tx256_sbp256_stable.pt")

def _sigma_tag_for_filename(value: float) -> str:
    value = float(value)
    text = f"{value:.6f}".rstrip("0").rstrip(".")
    if "." not in text:
        text = f"{text}.0"
    return text.replace("-", "m").replace(".", "p")

SMOOTHED_SIGMA_TAG = _sigma_tag_for_filename(GAUSSIAN_SMOOTHING_SIGMA_BINS)
STABLE_SSL_SESSION_STATS_PATH_SMOOTHED = Path(
    "/content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/contrastive/precomputed_ssl_session_stats/"
    f"session_feature_stats_session_featurewise_v1_refds000950_cap126682_tx256_sbp256_smooth_sigma{SMOOTHED_SIGMA_TAG}_stable.pt"
)

if SESSION_STATS_VARIANT == "smoothed":
    STABLE_SSL_SESSION_STATS_PATH = STABLE_SSL_SESSION_STATS_PATH_SMOOTHED
elif SESSION_STATS_VARIANT == "unsmoothed":
    STABLE_SSL_SESSION_STATS_PATH = STABLE_SSL_SESSION_STATS_PATH_UNSMOOTHED
elif SESSION_STATS_VARIANT == "none":
    STABLE_SSL_SESSION_STATS_PATH = None
else:
    raise ValueError("SESSION_STATS_VARIANT must be one of {'smoothed', 'unsmoothed', 'none'}")

if STABLE_SSL_SESSION_STATS_PATH is not None and not STABLE_SSL_SESSION_STATS_PATH.exists():
    print(
        "warning: selected STABLE_SSL_SESSION_STATS_PATH does not exist; "
        "falling back to recomputing session stats during cache preparation:",
        STABLE_SSL_SESSION_STATS_PATH,
    )
    STABLE_SSL_SESSION_STATS_PATH = None

SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH = "/content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/masked_reconstruction/colab_s5_masked_reconstruction_seg80_20260421T155922Z/checkpoints/step_014000_20260421T175024Z.pt"
SSL_RECOVERY_RUN_DIR = None
ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH = None

FEATURE_MODE = "tx_only"
SEGMENT_BINS = 80
PATCH_SIZE = 4
PATCH_STRIDE = 4
HIDDEN_SIZE = 256
S5_STATE_SIZE = 128
NUM_LAYERS = 4
DROPOUT = 0
BATCH_SIZE = 32
NUM_STEPS = 1500
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0
VAL_EVERY = 50
VAL_BATCHES = 10
CHECKPOINT_EVERY_STEPS = 500
DATASET_WEIGHT_ALPHA = 0.25
EXAMPLES_PER_SHARD = 8
POST_PROJ_NORM = "rms"
RECONSTRUCTION_HEAD_TYPE = "mlp"
BACKBONE_DIRECTION = "bidirectional"

MASK_UNIT = "patch"  # Set to "bin" for raw-bin contiguous masking.
MASK_TOKEN_PLACEMENT = "before_projection"
MASK_RATIO = 0.4
PATCH_SPAN_LENGTH_MIN = 2
PATCH_SPAN_LENGTH_MAX = 4
BIN_SPAN_LENGTH_MIN = PATCH_SPAN_LENGTH_MIN * PATCH_SIZE
BIN_SPAN_LENGTH_MAX = PATCH_SPAN_LENGTH_MAX * PATCH_SIZE
SPAN_LENGTH_MIN = PATCH_SPAN_LENGTH_MIN if MASK_UNIT == "patch" else BIN_SPAN_LENGTH_MIN
SPAN_LENGTH_MAX = PATCH_SPAN_LENGTH_MAX if MASK_UNIT == "patch" else BIN_SPAN_LENGTH_MAX
NUM_SPANS_MODE = "multiple"
ALLOW_BIN_FRACTIONAL_OVERLAP = True
RECONSTRUCT_TARGET = "raw_patch"
LOSS_MODE = "masked_only"

CACHE_ACCESS_MODE = "drive_direct"  # or "copy_to_local"
LOCAL_CACHE_BASE = "/content/utah_ssl_cache"
FORCE_RECOPY_LOCAL_CACHE = False
EXCLUDED_DATASETS = {"brain2text25"}
LOG_EVERY = 10

USE_NORMALIZATION = True
if USE_NORMALIZATION and GAUSSIAN_SMOOTHING_SIGMA_BINS > 0 and SESSION_STATS_VARIANT == "unsmoothed":
    print(
        "warning: using smoothed inputs with unsmoothed session stats; "
        "set SESSION_STATS_VARIANT='smoothed' for matched preprocessing."
    )

CACHE_ACCESS_CONFIG = CacheAccessConfig(
    mode=CACHE_ACCESS_MODE,
    local_cache_base=LOCAL_CACHE_BASE,
    force_recopy_local_cache=FORCE_RECOPY_LOCAL_CACHE,
    excluded_datasets=tuple(sorted(EXCLUDED_DATASETS)),
    seed=SEED,
    segment_bins=SEGMENT_BINS,
    use_normalization=USE_NORMALIZATION,
    gaussian_smoothing_sigma_bins=GAUSSIAN_SMOOTHING_SIGMA_BINS,
    examples_per_shard=EXAMPLES_PER_SHARD,
    feature_mode=FEATURE_MODE,
    boundary_key_mode="subject_if_available",
)

SSL_TRAINING_CONFIG = SSLTrainingConfig(
    seed=SEED,
    feature_mode=FEATURE_MODE,
    segment_bins=SEGMENT_BINS,
    patch_size=PATCH_SIZE,
    patch_stride=PATCH_STRIDE,
    hidden_size=HIDDEN_SIZE,
    s5_state_size=S5_STATE_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    batch_size=BATCH_SIZE,
    num_steps=NUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    val_every=VAL_EVERY,
    val_batches=VAL_BATCHES,
    checkpoint_every_steps=CHECKPOINT_EVERY_STEPS,
    dataset_weight_alpha=DATASET_WEIGHT_ALPHA,
    examples_per_shard=EXAMPLES_PER_SHARD,
    log_every=LOG_EVERY,
    post_proj_norm=POST_PROJ_NORM,
    reconstruction_head_type=RECONSTRUCTION_HEAD_TYPE,
    backbone_direction=BACKBONE_DIRECTION,
    boundary_key_mode="subject_if_available",
    mask_unit=MASK_UNIT,
    mask_token_placement=MASK_TOKEN_PLACEMENT,
    mask_ratio=MASK_RATIO,
    span_length_min=SPAN_LENGTH_MIN,
    span_length_max=SPAN_LENGTH_MAX,
    num_spans_mode=NUM_SPANS_MODE,
    reconstruct_target=RECONSTRUCT_TARGET,
    loss_mode=LOSS_MODE,
    allow_bin_fractional_overlap=ALLOW_BIN_FRACTIONAL_OVERLAP,
)

print("Notebook workflow switches:", {
    "SSL_STATE_MODE": SSL_STATE_MODE,
    "SESSION_STATS_VARIANT": SESSION_STATS_VARIANT,
    "STABLE_SSL_SESSION_STATS_PATH": STABLE_SSL_SESSION_STATS_PATH,
    "STABLE_SSL_SESSION_STATS_PATH_UNSMOOTHED": STABLE_SSL_SESSION_STATS_PATH_UNSMOOTHED,
    "STABLE_SSL_SESSION_STATS_PATH_SMOOTHED": STABLE_SSL_SESSION_STATS_PATH_SMOOTHED,
    "SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH": SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH,
    "SSL_RECOVERY_RUN_DIR": SSL_RECOVERY_RUN_DIR,
    "ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH": ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH,
})
print("Masking config:", {
    "mask_unit": MASK_UNIT,
    "mask_token_placement": MASK_TOKEN_PLACEMENT,
    "mask_ratio": MASK_RATIO,
    "span_length_min": SPAN_LENGTH_MIN,
    "span_length_max": SPAN_LENGTH_MAX,
    "num_spans_mode": NUM_SPANS_MODE,
})
print("Patch config:", {
    "patch_size": PATCH_SIZE,
    "patch_stride": PATCH_STRIDE,
    "segment_bins": SEGMENT_BINS,
    "reconstruction_head_type": RECONSTRUCTION_HEAD_TYPE,
    "backbone_direction": BACKBONE_DIRECTION,
})
print("Feature config:", {
    "feature_mode": FEATURE_MODE,
    "use_normalization": USE_NORMALIZATION,
    "gaussian_smoothing_sigma_bins": GAUSSIAN_SMOOTHING_SIGMA_BINS,
})
print("CACHE_ACCESS_CONFIG:", CACHE_ACCESS_CONFIG)
print("SSL_TRAINING_CONFIG:", SSL_TRAINING_CONFIG)




In [ ]:
# Resolve cache access mode, summarize datasets, and build the reusable cache context.

import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CACHE_CONTEXT = prepare_cache_context(
    cache_candidates=cache_candidates,
    config=CACHE_ACCESS_CONFIG,
)
FULL_DIM = CACHE_CONTEXT.full_dim

print("DEVICE:", DEVICE)
print("CACHE_CONTEXT cache_root:", CACHE_CONTEXT.cache_root)
print("FULL_DIM:", FULL_DIM)
print("session split summary:")
for dataset, summary in CACHE_CONTEXT.session_split_summary.items():
    print(
        f" - {dataset}: sessions={summary['total_sessions']} train_sessions={summary['train_sessions']} "
        f"val_sessions={summary['val_sessions']} train_examples={summary['train_examples']} "
        f"val_examples={summary['val_examples']}"
    )


In [ ]:
# Load precomputed SSL session-level featurewise z-scoring stats when available.

if STABLE_SSL_SESSION_STATS_PATH is not None:
    SSL_SESSION_STATS_STATE = load_precomputed_session_feature_stats_into_cache_context(
        cache_context=CACHE_CONTEXT,
        stats_path=STABLE_SSL_SESSION_STATS_PATH,
        
    )
    print("Loaded cached SSL session stats from stable path.")
else:
    SSL_SESSION_STATS_STATE = {
        "stats_path": None,
        "metadata": {
            "source": "prepare_cache_context",
            "gaussian_smoothing_sigma_bins": float(GAUSSIAN_SMOOTHING_SIGMA_BINS),
            "session_stats_bin_stride": int(EXPECTED_SESSION_STATS_BIN_STRIDE),
        },
        "session_count": len(CACHE_CONTEXT.session_feature_stats),
        "use_normalization": CACHE_CONTEXT.use_normalization,
    }
    print("Using session stats computed during cache preparation.")

print("session_count:", SSL_SESSION_STATS_STATE["session_count"])
print("use_normalization:", SSL_SESSION_STATS_STATE["use_normalization"])
print("metadata:", SSL_SESSION_STATS_STATE["metadata"])

print("expected_gaussian_smoothing_sigma_bins:", GAUSSIAN_SMOOTHING_SIGMA_BINS)
meta_sigma = SSL_SESSION_STATS_STATE["metadata"].get("gaussian_smoothing_sigma_bins")
print("stats_metadata_gaussian_smoothing_sigma_bins:", meta_sigma)
if meta_sigma is not None and abs(float(meta_sigma) - float(GAUSSIAN_SMOOTHING_SIGMA_BINS)) > 1e-6:
    print(
        "warning: loaded session stats metadata smoothing sigma does not match current config. "
        "This can skew normalization scale."
    )

print("expected_session_stats_bin_stride:", EXPECTED_SESSION_STATS_BIN_STRIDE)
meta_stride = SSL_SESSION_STATS_STATE["metadata"].get("session_stats_bin_stride")
print("stats_metadata_session_stats_bin_stride:", meta_stride)
meta_stride_norm = None
if meta_stride is not None:
    try:
        meta_stride_norm = int(round(float(meta_stride)))
    except (TypeError, ValueError):
        print(
            "warning: loaded session stats metadata stride is not parseable as a number. "
            "This can skew normalization scale."
        )
if meta_stride_norm is not None and int(meta_stride_norm) != int(EXPECTED_SESSION_STATS_BIN_STRIDE):
    print(
        "warning: loaded session stats metadata stride does not match current config. "
        "This can skew normalization scale."
    )





In [ ]:
# # Quick batch inspection.

# INSPECT_SAMPLER = build_segment_sampler(
#     CACHE_CONTEXT,
#     "train",
#     batch_size=4,
#     seed=SEED,
#     segment_bins=SEGMENT_BINS,
#     dataset_weight_alpha=DATASET_WEIGHT_ALPHA,
#     examples_per_shard=EXAMPLES_PER_SHARD,
# )
# inspect_batch = INSPECT_SAMPLER.sample_batch()
# print("inspect_batch[x].shape:", tuple(inspect_batch["x"].shape))
# print("inspect_batch[lengths]:", inspect_batch["lengths"].tolist())
# print("inspect_batch[datasets]:", inspect_batch["datasets"])
# print("inspect_batch[sessions]:", inspect_batch["sessions"])


In [ ]:
# Acquire SSL state: either train now or recover from a saved checkpoint.

if USE_NORMALIZATION and not CACHE_CONTEXT.session_feature_stats:
    raise RuntimeError("Run the stable session-stats cell first so session-level z-scoring is loaded into CACHE_CONTEXT.")

if SSL_STATE_MODE == "train":
    SSL_RUN_STATE = run_ssl_training(
        cache_context=CACHE_CONTEXT,
        config=SSL_TRAINING_CONFIG,
        output_root=OUTPUT_ROOT,
        device=DEVICE,
    )
elif SSL_STATE_MODE == "recover":
    resolved_checkpoint_path = resolve_ssl_checkpoint_path(
        output_root=OUTPUT_ROOT,
        explicit_checkpoint_path=SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH,
        run_dir=SSL_RECOVERY_RUN_DIR,
    )
    SSL_RUN_STATE = recover_ssl_run_state_from_checkpoint(
        checkpoint_path=resolved_checkpoint_path,
        cache_context=CACHE_CONTEXT,
        device=DEVICE,
        fallback_config=SSL_TRAINING_CONFIG,
    )
else:
    raise ValueError(f"Unsupported SSL_STATE_MODE: {SSL_STATE_MODE}")

print("run_name:", SSL_RUN_STATE["run_name"])
print("run_dir:", SSL_RUN_STATE["run_dir"])
print("checkpoint_path:", SSL_RUN_STATE["checkpoint_path"])
print("best_score:", SSL_RUN_STATE["best_score"])
print("best_step:", SSL_RUN_STATE["best_step"])


In [ ]:
# SSL RESUME cell: continue training from the current in-memory SSL_RUN_STATE.

RESUME_ADDITIONAL_STEPS = 1000

if "SSL_RUN_STATE" not in globals():
    if ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH is not None:
        resolved_checkpoint_path = Path(ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH)
        if not resolved_checkpoint_path.exists():
            raise FileNotFoundError(
                f"ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH does not exist: {resolved_checkpoint_path}"
            )
    else:
        resolved_checkpoint_path = resolve_ssl_checkpoint_path(
            output_root=OUTPUT_ROOT,
            explicit_checkpoint_path=SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH,
            run_dir=SSL_RECOVERY_RUN_DIR,
        )
    SSL_RUN_STATE = recover_ssl_run_state_from_checkpoint(
        checkpoint_path=resolved_checkpoint_path,
        cache_context=CACHE_CONTEXT,
        device=DEVICE,
        fallback_config=SSL_TRAINING_CONFIG,
    )
    print("Recovered SSL_RUN_STATE from checkpoint:", resolved_checkpoint_path)

SSL_RUN_STATE = resume_ssl_training(
    run_state=SSL_RUN_STATE,
    additional_steps=RESUME_ADDITIONAL_STEPS,
    cache_context=CACHE_CONTEXT,
    device=DEVICE,
)


In [ ]:
# Resolve the active checkpoint path used by downstream probe recovery.

if "SSL_RUN_STATE" not in globals() and ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH is None:
    resolved_checkpoint_path = resolve_ssl_checkpoint_path(
        output_root=OUTPUT_ROOT,
        explicit_checkpoint_path=SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH,
        run_dir=SSL_RECOVERY_RUN_DIR,
    )
    SSL_RUN_STATE = recover_ssl_run_state_from_checkpoint(
        checkpoint_path=resolved_checkpoint_path,
        cache_context=CACHE_CONTEXT,
        device=DEVICE,
        fallback_config=SSL_TRAINING_CONFIG,
    )
    print("Recovered SSL_RUN_STATE from checkpoint:", resolved_checkpoint_path)

if ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH is None:
    ACTIVE_SSL_CHECKPOINT_PATH = Path(SSL_RUN_STATE["checkpoint_path"])
else:
    ACTIVE_SSL_CHECKPOINT_PATH = Path(ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH)
    if not ACTIVE_SSL_CHECKPOINT_PATH.exists():
        raise FileNotFoundError(
            f"ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH does not exist: {ACTIVE_SSL_CHECKPOINT_PATH}"
        )

ACTIVE_SSL_CHECKPOINT_RUN_DIR = (
    ACTIVE_SSL_CHECKPOINT_PATH.parent.parent
    if ACTIVE_SSL_CHECKPOINT_PATH.parent.name == "checkpoints"
    else ACTIVE_SSL_CHECKPOINT_PATH.parent
)

print("ACTIVE_SSL_CHECKPOINT_PATH:", ACTIVE_SSL_CHECKPOINT_PATH)
print("ACTIVE_SSL_CHECKPOINT_RUN_DIR:", ACTIVE_SSL_CHECKPOINT_RUN_DIR)



In [ ]:
# Plot train / val curves and print the compact SSL reconstruction scorecard.

if "SSL_RUN_STATE" not in globals():
    resolved_checkpoint_path = resolve_ssl_checkpoint_path(
        output_root=OUTPUT_ROOT,
        explicit_checkpoint_path=SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH,
        run_dir=SSL_RECOVERY_RUN_DIR,
    )
    SSL_RUN_STATE = recover_ssl_run_state_from_checkpoint(
        checkpoint_path=resolved_checkpoint_path,
        cache_context=CACHE_CONTEXT,
        device=DEVICE,
        fallback_config=SSL_TRAINING_CONFIG,
    )
    print("Recovered SSL_RUN_STATE from checkpoint:", resolved_checkpoint_path)

SSL_RECON_SCORECARD = display_ssl_reconstruction_report(
    SSL_RUN_STATE,
    zero_baseline_df=globals().get("ZERO_BASELINE_DF"),
)


In [ ]:
# Optional: list saved SSL checkpoints for the active run.


CHECKPOINT_LIST_RUN_DIR = Path(ACTIVE_SSL_CHECKPOINT_RUN_DIR)
AVAILABLE_SSL_CHECKPOINTS = list_ssl_checkpoints(CHECKPOINT_LIST_RUN_DIR)
display(pd.DataFrame(AVAILABLE_SSL_CHECKPOINTS))


## Downstream Probe Workflow

These cells keep the same held-out `Brain2Text25` workflow as the other `s5` notebooks.
The main apples-to-apples frozen comparison is:

- pretrained masked-reconstruction encoder, frozen, linear probe
- random-init encoder, frozen, same linear probe


In [ ]:
# Configure and preview the held-out Brain2Text25 probe split.

PROBE_SESSION_LIMIT = 8
PROBE_TARGET_SESSION_COUNT = 4
PROBE_BATCH_SIZE = 8
PROBE_BUDGET_SECONDS = 240
PROBE_MAX_STEPS = 80
PROBE_HEAD_LEARNING_RATE = 1e-3
ENCODER_LEARNING_RATE = 3e-4
PROBE_WEIGHT_DECAY = 1e-2
PROBE_INPUT_TAIL_BINS = None  # Full sequence for CTC probes.

DOWNSTREAM_PROBE_CONFIG = DownstreamProbeConfig(
    enabled=True,
    seed=SEED,
    session_limit=PROBE_SESSION_LIMIT,
    target_session_count=PROBE_TARGET_SESSION_COUNT,
    probe_batch_size=PROBE_BATCH_SIZE,
    probe_budget_seconds=PROBE_BUDGET_SECONDS,
    max_probe_steps=PROBE_MAX_STEPS,
    probe_head_learning_rate=PROBE_HEAD_LEARNING_RATE,
    encoder_learning_rate=None,
    weight_decay=PROBE_WEIGHT_DECAY,
    probe_head_type="linear",
    probe_input_tail_bins=PROBE_INPUT_TAIL_BINS,
)

downstream_problem_preview = build_downstream_probe_problem(
    cache_root=Path(CACHE_CONTEXT.cache_root),
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    feature_mode=FEATURE_MODE,
)
print("eligible_sessions:", len(downstream_problem_preview["eligible_entries"]))
print("preview_source_sessions:", [entry.session_base for entry in downstream_problem_preview["split"].train])
print("preview_target_sessions:", [entry.session_base for entry in downstream_problem_preview["split"].val])
print("probe_input_tail_bins:", DOWNSTREAM_PROBE_CONFIG.probe_input_tail_bins)


In [ ]:
# Shared downstream experiment helpers.

FROZEN_LINEAR_PROBE_OVERRIDES = {
    "probe_head_type": "linear",
    "probe_input_tail_bins": PROBE_INPUT_TAIL_BINS,
    "max_probe_steps": PROBE_MAX_STEPS,
}
PROBE_HEAD_SPECS = [
    {"probe_head_type": "linear", "probe_input_tail_bins": PROBE_INPUT_TAIL_BINS},
]

def build_notebook_random_probe_state(*, seed_offset: int = 0):
    return build_random_init_probe_state(
        reference_config=DOWNSTREAM_PROBE_DEFAULT_STATE["checkpoint_config"],
        input_dim=DOWNSTREAM_PROBE_DEFAULT_STATE["input_dim"],
        seed=SEED + int(seed_offset),
        base_run_dir=DOWNSTREAM_PROBE_BASE_RUN_DIR,
    )


In [ ]:
# Recover the default SSL encoder and prepare reusable downstream probe state.

DOWNSTREAM_PROBE_DEFAULT_STATE = recover_downstream_probe_state(
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    output_root=OUTPUT_ROOT,
    input_dim=FULL_DIM,
    default_checkpoint_config=SSL_TRAINING_CONFIG.checkpoint_config(),
    in_memory_model=SSL_RUN_STATE["model"] if "SSL_RUN_STATE" in globals() else None,
    current_checkpoint_path=Path(ACTIVE_SSL_CHECKPOINT_PATH),
    current_run_dir=Path(ACTIVE_SSL_CHECKPOINT_RUN_DIR),
)

DOWNSTREAM_PROBE_BASE_RUN_DIR = Path(DOWNSTREAM_PROBE_DEFAULT_STATE["base_run_dir"])
print("DOWNSTREAM_PROBE_ENCODER_SOURCE:", DOWNSTREAM_PROBE_DEFAULT_STATE["source"])
print("DOWNSTREAM_PROBE_BASE_RUN_DIR:", DOWNSTREAM_PROBE_BASE_RUN_DIR)
print("DOWNSTREAM_PROBE_CHECKPOINT_PATH:", DOWNSTREAM_PROBE_DEFAULT_STATE["checkpoint_path"])


In [ ]:
# Frozen SSL checkpoint probes (linear).

SSL_CHECKPOINT_HEAD_SUMMARIES = run_probe_head_sweep(
    probe_state=DOWNSTREAM_PROBE_DEFAULT_STATE,
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    cache_root=Path(CACHE_CONTEXT.cache_root),
    device=DEVICE,
    variant_prefix="ssl_checkpoint",
    artifact_prefix="ssl_checkpoint",
    train_encoder=False,
    head_specs=PROBE_HEAD_SPECS,
)

SSL_CHECKPOINT_LINEAR_SUMMARY = next(
    summary for summary in SSL_CHECKPOINT_HEAD_SUMMARIES if summary.get("probe_head_type") == "linear"
)

display_probe_summaries(*SSL_CHECKPOINT_HEAD_SUMMARIES)


In [ ]:
# Random-init frozen probe baselines (linear).

RANDOM_INIT_FROZEN_STATE = build_notebook_random_probe_state(seed_offset=1000)
RANDOM_INIT_HEAD_SUMMARIES = run_probe_head_sweep(
    probe_state=RANDOM_INIT_FROZEN_STATE,
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    cache_root=Path(CACHE_CONTEXT.cache_root),
    device=DEVICE,
    variant_prefix="random_init",
    artifact_prefix="random_init",
    train_encoder=False,
    head_specs=PROBE_HEAD_SPECS,
)

RANDOM_INIT_LINEAR_SUMMARY = next(
    summary for summary in RANDOM_INIT_HEAD_SUMMARIES if summary.get("probe_head_type") == "linear"
)

display_probe_summaries(*SSL_CHECKPOINT_HEAD_SUMMARIES, *RANDOM_INIT_HEAD_SUMMARIES)


In [ ]:

B2T24_PROBE_DATASET = "brain2text24"
if "B2T24_SSL_SPLIT_INFO" in globals():
    B2T24_CONFIG_TRAIN_SESSION_IDS = [str(session_id) for session_id in B2T24_SSL_SPLIT_INFO.get("train_session_ids", [])]
    B2T24_CONFIG_VAL_SESSION_IDS = [str(session_id) for session_id in B2T24_SSL_SPLIT_INFO.get("val_session_ids", [])]
else:
    split_summary = CACHE_CONTEXT.session_split_summary.get(B2T24_PROBE_DATASET, {})
    B2T24_CONFIG_TRAIN_SESSION_IDS = [str(session_id) for session_id in split_summary.get("train_session_ids", [])]
    B2T24_CONFIG_VAL_SESSION_IDS = [str(session_id) for session_id in split_summary.get("val_session_ids", [])]
    if not B2T24_CONFIG_VAL_SESSION_IDS:
        split_rows = CACHE_CONTEXT.split_rows_by_dataset
        train_rows = split_rows.get("train", {}).get(B2T24_PROBE_DATASET, [])
        val_rows = split_rows.get("val", {}).get(B2T24_PROBE_DATASET, [])
        B2T24_CONFIG_TRAIN_SESSION_IDS = sorted({str(row.session_id) for row in train_rows})
        B2T24_CONFIG_VAL_SESSION_IDS = sorted({str(row.session_id) for row in val_rows})

if not B2T24_CONFIG_VAL_SESSION_IDS:
    raise RuntimeError(
        "No Brain2Text24 val sessions were found in the active config split. "
        "Populate B2T24_SSL_SPLIT_INFO or run the split-config cell first."
    )

print("Brain2Text24 probe train sessions:", B2T24_CONFIG_TRAIN_SESSION_IDS)
print("Brain2Text24 probe val sessions:", B2T24_CONFIG_VAL_SESSION_IDS)


In [ ]:
# Brain2Text24 pooled val-session probes using the configured SSL split.

B2T24_SSL_CHECKPOINT_HEAD_SUMMARIES = run_probe_head_sweep(
    probe_state=DOWNSTREAM_PROBE_DEFAULT_STATE,
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    cache_root=Path(CACHE_CONTEXT.cache_root),
    device=DEVICE,
    variant_prefix="ssl_checkpoint_b2t24_config_val",
    artifact_prefix="ssl_checkpoint_b2t24_config_val",
    train_encoder=False,
    probe_dataset=B2T24_PROBE_DATASET,
    source_session_ids=B2T24_CONFIG_TRAIN_SESSION_IDS,
    target_session_ids=B2T24_CONFIG_VAL_SESSION_IDS,
    head_specs=PROBE_HEAD_SPECS,
)

B2T24_SSL_CHECKPOINT_LINEAR_SUMMARY = next(
    summary for summary in B2T24_SSL_CHECKPOINT_HEAD_SUMMARIES if summary.get("probe_head_type") == "linear"
)

B2T24_RANDOM_INIT_FROZEN_STATE = build_notebook_random_probe_state(seed_offset=2000)
B2T24_RANDOM_INIT_HEAD_SUMMARIES = run_probe_head_sweep(
    probe_state=B2T24_RANDOM_INIT_FROZEN_STATE,
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    cache_root=Path(CACHE_CONTEXT.cache_root),
    device=DEVICE,
    variant_prefix="random_init_b2t24_config_val",
    artifact_prefix="random_init_b2t24_config_val",
    train_encoder=False,
    probe_dataset=B2T24_PROBE_DATASET,
    source_session_ids=B2T24_CONFIG_TRAIN_SESSION_IDS,
    target_session_ids=B2T24_CONFIG_VAL_SESSION_IDS,
    head_specs=PROBE_HEAD_SPECS,
)

B2T24_RANDOM_INIT_LINEAR_SUMMARY = next(
    summary for summary in B2T24_RANDOM_INIT_HEAD_SUMMARIES if summary.get("probe_head_type") == "linear"
)

display_probe_summaries(
    *B2T24_SSL_CHECKPOINT_HEAD_SUMMARIES,
    *B2T24_RANDOM_INIT_HEAD_SUMMARIES,
)


In [ ]:
# Loss curves for B2T24 probe sweeps (SSL checkpoint vs random init)

from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt


def _load_probe_progress(summary: dict, run_label: str) -> pd.DataFrame:
    progress_path = Path(summary["progress_log_path"])
    if not progress_path.exists():
        print(f"[warn] missing progress log: {progress_path}")
        return pd.DataFrame()

    rows = []
    with progress_path.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            rec["run_label"] = run_label
            rec["model_variant"] = summary.get("model_variant")
            rec["probe_head_type"] = summary.get("probe_head_type")
            rows.append(rec)

    return pd.DataFrame(rows)


def plot_probe_loss_curves(
    ssl_summaries: list[dict],
    random_summaries: list[dict],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    progress_df = pd.concat(
        [
            *[_load_probe_progress(s, "SSL checkpoint") for s in ssl_summaries],
            *[_load_probe_progress(s, "Random init") for s in random_summaries],
        ],
        ignore_index=True,
    )

    if progress_df.empty:
        print("No probe progress records found.")
        return pd.DataFrame(), pd.DataFrame()

    train_df = progress_df[progress_df["event"] == "probe_train_report"].copy()
    final_df = progress_df[progress_df["event"] == "probe_session_complete"].copy()

    heads = sorted(set(train_df["probe_head_type"].dropna()) | set(final_df["probe_head_type"].dropna()))
    if not heads:
        heads = ["unknown"]

    fig, axes = plt.subplots(1, len(heads), figsize=(7 * len(heads), 4), squeeze=False)
    axes = axes[0]

    colors = {"SSL checkpoint": "C0", "Random init": "C1"}

    for ax, head in zip(axes, heads):
        for run_label in ["SSL checkpoint", "Random init"]:
            color = colors[run_label]

            t = train_df[(train_df["probe_head_type"] == head) & (train_df["run_label"] == run_label)].sort_values("step")
            if not t.empty:
                ax.plot(t["step"], t["train_ctc_bpphone"], color=color, lw=2, label=f"{run_label} train")

            v = final_df[(final_df["probe_head_type"] == head) & (final_df["run_label"] == run_label)].sort_values("step")
            if not v.empty:
                ax.scatter(
                    v["step"].iloc[-1],
                    v["val_ctc_bpphone"].iloc[-1],
                    color=color,
                    marker="X",
                    s=90,
                    label=f"{run_label} val (final)",
                )

        ax.set_title(f"Probe head: {head}")
        ax.set_xlabel("Probe step")
        ax.set_ylabel("CTC bits/phoneme (lower is better)")
        ax.grid(alpha=0.25)
        ax.legend()

    plt.tight_layout()
    plt.show()

    summary_table = (
        final_df[["run_label", "probe_head_type", "step", "val_ctc_bpphone", "val_phoneme_error_rate", "model_variant"]]
        .sort_values(["probe_head_type", "run_label"])
        .reset_index(drop=True)
    )
    display(summary_table)

    return train_df, final_df


# Use all heads from your sweep:
B2T24_PROBE_TRAIN_DF, B2T24_PROBE_FINAL_DF = plot_probe_loss_curves(
    B2T24_SSL_CHECKPOINT_HEAD_SUMMARIES,
    B2T24_RANDOM_INIT_HEAD_SUMMARIES,
)

# If you only want linear:
# B2T24_PROBE_TRAIN_DF, B2T24_PROBE_FINAL_DF = plot_probe_loss_curves(
#     [B2T24_SSL_CHECKPOINT_LINEAR_SUMMARY],
#     [B2T24_RANDOM_INIT_LINEAR_SUMMARY],
# )


In [ ]:
# Probe suite utilities for comparing multiple SSL checkpoints.

MAIN_SUMMARIES = [
    globals().get("SSL_CHECKPOINT_LINEAR_SUMMARY"),
    globals().get("RANDOM_INIT_LINEAR_SUMMARY"),
]
display_probe_summaries(*MAIN_SUMMARIES)

CHECKPOINT_PROBE_PATHS = [Path(ACTIVE_SSL_CHECKPOINT_PATH)]

# Add extra checkpoints as needed, e.g.:
# CHECKPOINT_PROBE_PATHS = [
#     Path(ACTIVE_SSL_CHECKPOINT_PATH),
#     Path("/content/.../checkpoints/step_006000_...pt"),
#     Path("/content/.../checkpoints/step_008000_...pt"),
# ]

RUN_CHECKPOINT_PROBE_SUITE = False

if RUN_CHECKPOINT_PROBE_SUITE:
    CHECKPOINT_PROBE_SUMMARIES = run_checkpoint_probe_suite(
        checkpoint_paths=[Path(path) for path in CHECKPOINT_PROBE_PATHS],
        probe_config=DOWNSTREAM_PROBE_CONFIG,
        output_root=Path(OUTPUT_ROOT),
        input_dim=int(FULL_DIM),
        default_checkpoint_config=SSL_TRAINING_CONFIG.checkpoint_config(),
        cache_root=Path(CACHE_CONTEXT.cache_root),
        device=DEVICE,
        head_specs=PROBE_HEAD_SPECS,
        variant_prefix="ssl_checkpoint",
        artifact_prefix="ssl_checkpoint",
        train_encoder=False,
    )
    CHECKPOINT_PROBE_DF = display_probe_summaries(*CHECKPOINT_PROBE_SUMMARIES)
    display(CHECKPOINT_PROBE_DF)


In [ ]:
from dataclasses import replace
from pathlib import Path
import re
import pandas as pd

# --- Config ---
PROBE_SEEDS = [7, 17, 27]                    # 3 random seeds
CHECKPOINT_STEPS = [10_000, 14_000]          # evaluate these SSL checkpoints
SEED_HEAD_SPECS = [{"probe_head_type": "linear"}]  # keep to one probe head; change if needed

# Reuse your existing B2T24 split variables from prior cells.
if "B2T24_PROBE_DATASET" not in globals():
    B2T24_PROBE_DATASET = "brain2text24"

# Determine run/checkpoint roots from your current probe state.
SSL_RUN_DIR = Path(DOWNSTREAM_PROBE_DEFAULT_STATE["base_run_dir"])
CHECKPOINTS_DIR = SSL_RUN_DIR / "checkpoints"
if not CHECKPOINTS_DIR.exists():
    raise FileNotFoundError(f"Checkpoint dir not found: {CHECKPOINTS_DIR}")

def _extract_step_from_checkpoint_path(path: Path) -> int | None:
    # Supports: step_014000_*.pt and checkpoint_final_step_014000_*.pt
    m = re.search(r"(?:^step_|checkpoint_final_step_)(\d+)", path.stem)
    return int(m.group(1)) if m else None

def _resolve_checkpoint_for_step(checkpoints_dir: Path, step: int) -> Path:
    all_ckpts = sorted(checkpoints_dir.glob("*.pt"))
    matches = [p for p in all_ckpts if _extract_step_from_checkpoint_path(p) == int(step)]
    if not matches:
        raise FileNotFoundError(
            f"No checkpoint found for step={step} in {checkpoints_dir}. "
            f"Available: {[p.name for p in all_ckpts[:10]]} ..."
        )
    return matches[-1]  # latest lexicographic (usually latest timestamped file)

base_probe_config = replace(DOWNSTREAM_PROBE_CONFIG, explicit_checkpoint_path=None)

seeded_probe_summaries = []

for step in CHECKPOINT_STEPS:
    ckpt_path = _resolve_checkpoint_for_step(CHECKPOINTS_DIR, step)
    print(f"\nCheckpoint step {step}: {ckpt_path.name}")

    for seed in PROBE_SEEDS:
        seeded_probe_config = replace(
            base_probe_config,
            seed=int(seed),
            explicit_checkpoint_path=str(ckpt_path),
        )

        seeded_probe_state = recover_downstream_probe_state(
            probe_config=seeded_probe_config,
            output_root=Path(OUTPUT_ROOT),
            input_dim=int(DOWNSTREAM_PROBE_DEFAULT_STATE["input_dim"]),
            default_checkpoint_config=SSL_TRAINING_CONFIG.checkpoint_config(),
            in_memory_model=None,
            current_checkpoint_path=ckpt_path,
            current_run_dir=SSL_RUN_DIR,
        )

        run_summaries = run_probe_head_sweep(
            probe_state=seeded_probe_state,
            probe_config=seeded_probe_config,
            cache_root=Path(CACHE_CONTEXT.cache_root),
            device=DEVICE,
            variant_prefix=f"ssl_step{step:05d}_seed{seed}",
            artifact_prefix=f"ssl_step{step:05d}_seed{seed}",
            train_encoder=False,
            probe_dataset=B2T24_PROBE_DATASET,
            source_session_ids=B2T24_CONFIG_TRAIN_SESSION_IDS,
            target_session_ids=B2T24_CONFIG_VAL_SESSION_IDS,
            head_specs=SEED_HEAD_SPECS,
        )

        for s in run_summaries:
            s["seed"] = int(seed)
            s["checkpoint_step"] = int(step)
            s["evaluated_checkpoint_path"] = str(ckpt_path)
        seeded_probe_summaries.extend(run_summaries)

# Save raw summaries for later reuse
B2T24_SEEDED_CHECKPOINT_PROBE_SUMMARIES = seeded_probe_summaries

# Raw per-run table
seeded_df = pd.DataFrame(B2T24_SEEDED_CHECKPOINT_PROBE_SUMMARIES)
display(
    seeded_df[
        [
            "checkpoint_step",
            "seed",
            "probe_head_type",
            "val_ctc_bpphone",
            "val_phoneme_error_rate",
            "model_variant",
        ]
    ].sort_values(["checkpoint_step", "probe_head_type", "seed"]).reset_index(drop=True)
)

# Averaged results across seeds
B2T24_CHECKPOINT_SEED_AVG = (
    seeded_df.groupby(["checkpoint_step", "probe_head_type"], dropna=False)
    .agg(
        n_runs=("val_ctc_bpphone", "size"),
        mean_val_ctc_bpphone=("val_ctc_bpphone", "mean"),
        std_val_ctc_bpphone=("val_ctc_bpphone", "std"),
        mean_val_per=("val_phoneme_error_rate", "mean"),
        std_val_per=("val_phoneme_error_rate", "std"),
    )
    .reset_index()
    .sort_values(["checkpoint_step", "probe_head_type"])
)

display(B2T24_CHECKPOINT_SEED_AVG)


## Notes

This notebook intentionally omits the contrastive retrieval diagnostics from the older `s5` notebooks.
For this experiment, the primary scorecard is held-out frozen phoneme transfer, with masked-reconstruction loss used as a training diagnostic.


In [ ]:
# Frozen pretrained backbone + encoder-decoder CTC probe head
# 80 steps, with slightly higher LR on the encoder sub-head.

from dataclasses import replace
from pathlib import Path
import copy
import importlib
import math

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

probe_mod = importlib.import_module("masked_ssl.probe")

# --------------------------
# Config
# --------------------------
ENCDEC_SEED = 7
ENCDEC_STEPS = 80
ENCDEC_BATCH_SIZE = int(DOWNSTREAM_PROBE_CONFIG.probe_batch_size)
ENCDEC_WEIGHT_DECAY = float(DOWNSTREAM_PROBE_CONFIG.weight_decay)

ENCDEC_HIDDEN = 128
ENCDEC_DROPOUT = 0.1

ENCODER_HEAD_LR = 2e-3   # slightly higher
DECODER_LR = 1e-3

VAL_EVERY = 10

# --------------------------
# Custom probe head
# --------------------------
class EncoderDecoderCTCProbe(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, vocab_size: int, dropout: float = 0.1):
        super().__init__()
        self.encoder_head = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.norm = nn.LayerNorm(hidden_size * 2)
        self.dropout = nn.Dropout(dropout)
        self.decoder_head = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, hidden: torch.Tensor) -> torch.Tensor:
        z, _ = self.encoder_head(hidden)
        z = self.norm(z)
        z = self.dropout(z)
        return self.decoder_head(z)

# --------------------------
# Build problem + loaders (same split you already configured)
# --------------------------
encdec_cfg = replace(
    DOWNSTREAM_PROBE_CONFIG,
    seed=ENCDEC_SEED,
    max_probe_steps=ENCDEC_STEPS,
    probe_batch_size=ENCDEC_BATCH_SIZE,
)

problem = probe_mod.build_downstream_probe_problem(
    cache_root=Path(CACHE_CONTEXT.cache_root),
    probe_config=encdec_cfg,
    feature_mode=str(DOWNSTREAM_PROBE_DEFAULT_STATE.get("feature_mode", "tx_only")),
    boundary_key_mode=str(DOWNSTREAM_PROBE_DEFAULT_STATE.get("boundary_key_mode", "session")),
    dataset=str(B2T24_PROBE_DATASET),
    source_session_ids=B2T24_CONFIG_TRAIN_SESSION_IDS,
    target_session_ids=B2T24_CONFIG_VAL_SESSION_IDS,
)

target_stats_mode = "global" if len(problem["target_session_ids"]) == 1 else "per_session"
target_stats = probe_mod.compute_feature_stats(
    problem["target_train_rows"],
    cache_root=Path(problem["cache_root"]),
    mode=target_stats_mode,
    feature_mode=str(problem["feature_mode"]),
)

input_tail_bins = (
    int(encdec_cfg.probe_input_tail_bins)
    if encdec_cfg.probe_input_tail_bins is not None
    else None
)

train_loader = DataLoader(
    probe_mod.CanonicalSequenceDataset(
        problem["target_train_rows"],
        cache_root=Path(problem["cache_root"]),
        stats=target_stats,
        feature_mode=str(problem["feature_mode"]),
        boundary_key_mode=str(problem.get("boundary_key_mode", "session")),
        dataset=str(problem.get("dataset", B2T24_PROBE_DATASET)),
        input_tail_bins=input_tail_bins,
    ),
    **probe_mod._loader_kwargs(
        DEVICE,
        int(encdec_cfg.probe_batch_size),
        shuffle=True,
        collate_fn=probe_mod.collate_sequence_batch,
    ),
    generator=probe_mod._make_loader_generator(int(encdec_cfg.seed)),
)

val_loader = DataLoader(
    probe_mod.CanonicalSequenceDataset(
        problem["target_val_rows"],
        cache_root=Path(problem["cache_root"]),
        stats=target_stats,
        feature_mode=str(problem["feature_mode"]),
        boundary_key_mode=str(problem.get("boundary_key_mode", "session")),
        dataset=str(problem.get("dataset", B2T24_PROBE_DATASET)),
        input_tail_bins=input_tail_bins,
    ),
    **probe_mod._loader_kwargs(
        DEVICE,
        int(encdec_cfg.probe_batch_size),
        shuffle=False,
        collate_fn=probe_mod.collate_sequence_batch,
    ),
    generator=probe_mod._make_loader_generator(int(encdec_cfg.seed) + 1),
)

# --------------------------
# Model pieces
# --------------------------
with torch.random.fork_rng(devices=[]):
    torch.manual_seed(int(encdec_cfg.seed))
    frozen_encoder = copy.deepcopy(DOWNSTREAM_PROBE_DEFAULT_STATE["encoder"]).to(DEVICE)

for p in frozen_encoder.parameters():
    p.requires_grad = False
frozen_encoder.eval()

with torch.random.fork_rng(devices=[]):
    torch.manual_seed(int(encdec_cfg.seed) + 2)
    encdec_probe = EncoderDecoderCTCProbe(
        input_size=int(frozen_encoder.hidden_size),
        hidden_size=ENCDEC_HIDDEN,
        vocab_size=int(problem["vocab"]["num_classes"]),
        dropout=ENCDEC_DROPOUT,
    ).to(DEVICE)

with torch.random.fork_rng(devices=[]):
    torch.manual_seed(int(encdec_cfg.seed) + 3)
    target_affines = probe_mod.SessionLinearBank(
        tuple(problem["target_session_ids"]),
        int(frozen_encoder.token_dim),
    ).to(DEVICE)

enc_params = (
    list(encdec_probe.encoder_head.parameters())
    + list(encdec_probe.norm.parameters())
    + list(encdec_probe.dropout.parameters())
)
dec_params = list(encdec_probe.decoder_head.parameters())
affine_params = [p for p in target_affines.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW(
    [
        {"params": enc_params, "lr": ENCODER_HEAD_LR},
        {"params": dec_params, "lr": DECODER_LR},
        {"params": affine_params, "lr": DECODER_LR},
    ],
    lr=DECODER_LR,
    weight_decay=ENCDEC_WEIGHT_DECAY,
)

# --------------------------
# Train loop (fixed 80 steps)
# --------------------------
train_curve = []
val_curve = []

train_iter = iter(train_loader)
blank_index = int(problem["vocab"]["blank_index"])
clip_params = [p for p in encdec_probe.parameters() if p.requires_grad] + affine_params

for step in range(1, ENCDEC_STEPS + 1):
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        batch = next(train_iter)

    encdec_probe.train()
    target_affines.train()
    frozen_encoder.eval()

    x = batch["x"].to(DEVICE)
    input_lengths = batch["input_lengths"].to(DEVICE)
    labels = batch["labels"].to(DEVICE)
    label_lengths = batch["label_lengths"].to(DEVICE)

    outputs = frozen_encoder.encode(
        x,
        input_lengths,
        batch["session_ids"],
        use_source_affines=False,
        target_affines=target_affines,
    )
    logits = encdec_probe(outputs.hidden)

    loss_sum, target_count = probe_mod.compute_ctc_loss_sum(
        logits,
        outputs.token_lengths,
        labels,
        label_lengths,
        blank_index=blank_index,
    )
    if target_count <= 0:
        continue

    loss = loss_sum / target_count
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(clip_params, max_norm=1.0)
    optimizer.step()

    train_bpphone = float(loss.item()) / math.log(2.0)
    train_curve.append({"step": step, "train_ctc_bpphone": train_bpphone})

    if step == 1 or step % VAL_EVERY == 0 or step == ENCDEC_STEPS:
        metrics = probe_mod.evaluate_probe_session_metrics(
            encoder=frozen_encoder,
            probe_head=encdec_probe,
            target_affines=target_affines,
            loader=val_loader,
            device=DEVICE,
            blank_index=blank_index,
        )
        val_curve.append(
            {
                "step": step,
                "val_ctc_bpphone": float(metrics["val_ctc_bpphone"]),
                "val_per": float(metrics["val_phoneme_error_rate"]),
            }
        )
        print(
            f"step={step:03d} train_ctc={train_bpphone:.4f} "
            f"val_ctc={metrics['val_ctc_bpphone']:.4f} val_PER={100.0*metrics['val_phoneme_error_rate']:.2f}%"
        )

# --------------------------
# Plot
# --------------------------
train_df = pd.DataFrame(train_curve)
val_df = pd.DataFrame(val_curve)

plt.figure(figsize=(8, 4.5))
plt.plot(train_df["step"], train_df["train_ctc_bpphone"], label="train CTC bpphone")
if not val_df.empty:
    plt.plot(val_df["step"], val_df["val_ctc_bpphone"], marker="o", label="val CTC bpphone")
plt.xlabel("step")
plt.ylabel("CTC bits/phoneme")
plt.title("Frozen Pretrained Backbone + Encoder-Decoder Probe (80 steps)")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

display(val_df)
ENCDEC_PROBE_TRAIN_DF = train_df
ENCDEC_PROBE_VAL_DF = val_df
